# 최종 ML 파이프라인 v2 — 브랜드별 공실 추천

**변경점 (v1 → v2)**
- `업종구분`(일반/프랜차이즈 2종) → **`brand`**(일반카페 + 13개 프랜차이즈, 14종)로 교체
  → AUC 개선 확인됨 (3년 0.692→0.774, 5년 0.722→0.795, 8년 0.610→0.746)
- **위치 점수**(브랜드 무관 기본 추천): `brand='일반카페'`로 예측
- **브랜드별 점수**: `brand=선택브랜드`로 동일 모델 재예측 (모델 재학습 불필요, predict만 다시)
- 13개 브랜드 × 상위 100개를 미리 계산해 MVP용 `brand_top100.json` 생성 (Step J)

**변수 4그룹 + brand**
1. 위치 이력: 입점전_교체횟수 / 입점전_공실횟수 / 입점전_생존분기수
2. 물리 특성: 층구분 / 지하철거리_m / 건물내위치수 / 연면적 / 도로유형
3. 상권 맥락: 상권단계(잠재/안정/활성/침체) / 공실_상권대비 / 교체_상권대비 / TRDAR_SE_1
4. 경쟁 밀도: 경쟁자_수_입점시점 / 경쟁자_증가율 (2년전 대비)
5. **brand**: 일반카페 + 13개 프랜차이즈 브랜드명

**모델**: CatBoost (depth=2, l2_leaf_reg=15, iterations=150)

**공실 적용 시 보정**:
- B: 입점전 변수를 "마지막 영업 시점 기준"으로 재계산
- A: 3/5/8년 생존확률에 단조감소 제약 적용

**전제**: `pivot`(pivot_df), `cafe_history`, `viz`(viz_map_v2_clean), `result`(result_final_clean), `sangkwon_timeseries`가 메모리에 있음

## 0. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

quarters       = list(pivot.columns)
quarter_nums   = [int(q.replace('df_', '')) for q in quarters]
quarter_to_idx = {n: i for i, n in enumerate(quarter_nums)}

RADIUS_M = 500
EARTH_R  = 6371000
RADIUS_RAD = RADIUS_M / EARTH_R

print('준비 완료')

## A. 입점 이력 구축 (일반카페 + 프랜차이즈 통합)
- 가게(상가업소번호) 단위 입점/퇴점 분기 집계
- 라벨: 단기폐업 / 중기폐업 / 장수 / 제외
- 201512 입점 제외 (동어반복 차단)
- `brand` 컬럼: 일반카페는 '일반카페', 프랜차이즈는 브랜드명 그대로 사용

In [ ]:
cafe = cafe_history[cafe_history['brand'] != '스타벅스'].copy()
cafe['period_num'] = cafe['period'].str.replace('df_', '').astype(int)

spans = cafe.groupby(['위치ID', '상가업소번호']).agg(
    입점분기=('period_num', 'min'),
    퇴점분기=('period_num', 'max'),
    생존분기수=('period_num', 'count'),
    brand=('brand', 'first')
).reset_index()

spans['현재영업중'] = (spans['퇴점분기'] == 202512).astype(int)

def label(row):
    n, 현재 = row['생존분기수'], row['현재영업중']
    if 현재 == 1:
        return '장수' if n >= 32 else '제외'
    if n <= 8:  return '단기폐업'
    if n <= 32: return '중기폐업'
    return '장수'

spans['라벨'] = spans.apply(label, axis=1)

df = spans[(spans['라벨'] != '제외') & (spans['입점분기'] > 201512)].copy()
print(f'분석 대상: {len(df)}개')
print(df['brand'].value_counts())
print(df['라벨'].value_counts())

## A-2. 입점전 교체횟수 / 공실횟수 / 생존분기수

In [ ]:
def calc_pre_entry(위치ID, 입점분기):
    if 위치ID not in pivot.index:
        return np.nan, np.nan, np.nan
    row   = pivot.loc[위치ID]
    pre_q = [q for q, n in zip(quarters, quarter_nums) if n < 입점분기]
    if not pre_q:
        return np.nan, np.nan, np.nan
    vals  = row[pre_q].values
    공실  = int(pd.isna(vals).sum())
    생존  = int(pd.notna(vals).sum())
    교체  = 0
    prev  = vals[0]
    for v in vals[1:]:
        if pd.isna(prev) != pd.isna(v):
            교체 += 1
        elif not pd.isna(prev) and not pd.isna(v) and prev != v:
            교체 += 1
        prev = v
    return 교체, 공실, 생존

print('계산 중...')
pre = df.apply(lambda r: calc_pre_entry(r['위치ID'], r['입점분기']), axis=1)
df['입점전_교체횟수']   = [x[0] for x in pre]
df['입점전_공실횟수']   = [x[1] for x in pre]
df['입점전_생존분기수'] = [x[2] for x in pre]
df = df.dropna(subset=['입점전_교체횟수'])
print(f'완료: {len(df)}개')

## B. 물리 특성 조인

In [ ]:
df = df.merge(viz[['위치ID','층구분','지하철거리_m','건물내위치수',
                    '연면적','도로유형','위도','경도']], on='위치ID', how='left')

loc_coords_bak = cafe.dropna(subset=['위도','경도']).drop_duplicates('위치ID')\
                     .set_index('위치ID')[['위도','경도']]
df['위도'] = df['위도'].fillna(df['위치ID'].map(loc_coords_bak['위도']))
df['경도'] = df['경도'].fillna(df['위치ID'].map(loc_coords_bak['경도']))

df['층구분'] = df['층구분'].fillna('층정보없음')
for col in ['지하철거리_m','건물내위치수','연면적']:
    df[col] = df[col].fillna(df[col].median())
df['도로유형'] = df['도로유형'].fillna(df['도로유형'].mode()[0])

print(df[['층구분','지하철거리_m','건물내위치수','연면적','도로유형','위도','경도']].isnull().sum())

## C. 상권 맥락 조인 + 상권단계 계산
- `result_final`에서 TRDAR_SE_1, 상권평균교체/공실, 공실/교체_상권대비
- `sangkwon_timeseries`로 입점 시점 기준 상권단계(잠재/안정/활성/침체) 계산

In [ ]:
df = df.merge(result[['위치ID','TRDAR_CD','TRDAR_CD_N','TRDAR_SE_1',
                       '상권평균교체','상권평균공실',
                       '공실_상권대비','교체_상권대비']], on='위치ID', how='left')

for col in ['상권평균교체','상권평균공실','공실_상권대비','교체_상권대비']:
    df[col] = df[col].fillna(0)
df['TRDAR_SE_1'] = df['TRDAR_SE_1'].fillna('정보없음')

print(f'TRDAR_CD 결측: {df["TRDAR_CD"].isnull().sum()} / {len(df)}')

In [ ]:
def to_period(q):
    if q <= 201612: return '2015-2016'
    if q <= 201812: return '2017-2018'
    if q <= 202012: return '2019-2020'
    if q <= 202212: return '2021-2022'
    return '2023-2025'

thresholds = sangkwon_timeseries.groupby('period')[['평균교체','평균공실']].median()
sang_idx   = sangkwon_timeseries.set_index(['TRDAR_CD','period'])

def get_stage(trdar_cd, period):
    key = (trdar_cd, period)
    if pd.isna(trdar_cd) or key not in sang_idx.index:
        return '정보없음'
    교체, 공실 = sang_idx.loc[key, ['평균교체','평균공실']]
    th_교체, th_공실 = thresholds.loc[period]
    if 교체 < th_교체 and 공실 < th_공실: return '안정'
    if 교체 < th_교체 and 공실 >= th_공실: return '잠재'
    if 교체 >= th_교체 and 공실 < th_공실: return '활성'
    return '침체'

df['입점period'] = df['입점분기'].apply(to_period)
df['상권단계']   = df.apply(lambda r: get_stage(r['TRDAR_CD'], r['입점period']), axis=1)

print(df['상권단계'].value_counts())

## D. 경쟁 밀도 (반경 500m, 입점 시점 기준)
- 분기별 BallTree (haversine)
- 입점 시점 / 2년전(8분기 전) 경쟁자 수 → 증가율
- `add_competitor_density` 함수로 학습/공실 양쪽에 재사용

In [ ]:
coord_df = cafe.dropna(subset=['위도','경도'])
trees = {}
for q in coord_df['period'].unique():
    sub = coord_df[coord_df['period'] == q]
    trees[q] = BallTree(np.radians(sub[['위도','경도']].values), metric='haversine')

print(f'{len(trees)}개 분기 BallTree 생성 완료')

In [ ]:
def add_competitor_density(target_df, q_col='입점분기'):
    target_df['경쟁자_수_입점시점'] = np.nan
    target_df['경쟁자_수_2년전']   = np.nan

    for q_num in target_df[q_col].unique():
        mask = (target_df[q_col] == q_num) & target_df['위도'].notna()
        if mask.sum() == 0:
            continue
        pts = np.radians(target_df.loc[mask, ['위도','경도']].values)

        q_key = f'df_{q_num}'
        if q_key in trees:
            target_df.loc[mask, '경쟁자_수_입점시점'] = \
                trees[q_key].query_radius(pts, r=RADIUS_RAD, count_only=True)

        idx = quarter_to_idx.get(q_num)
        if idx is not None and idx - 8 >= 0:
            q_key_past = quarters[idx - 8]
            if q_key_past in trees:
                target_df.loc[mask, '경쟁자_수_2년전'] = \
                    trees[q_key_past].query_radius(pts, r=RADIUS_RAD, count_only=True)

    target_df['경쟁자_수_입점시점'] = target_df['경쟁자_수_입점시점'].fillna(0)
    target_df['경쟁자_수_2년전']   = target_df['경쟁자_수_2년전'].fillna(0)

    cur, past = target_df['경쟁자_수_입점시점'], target_df['경쟁자_수_2년전']
    target_df['경쟁자_증가율'] = np.where(
        past == 0, np.where(cur > 0, 1.0, 0.0), (cur - past) / past
    )
    return target_df

df = add_competitor_density(df, q_col='입점분기')
print(df[['경쟁자_수_입점시점','경쟁자_수_2년전','경쟁자_증가율']].describe())

## E. 피처 정의 + CatBoost 3/5/8년차 모델
**`brand`를 피처로 사용** (일반카페 + 13개 프랜차이즈, 14종) — 표본 공유로 AUC 향상

depth=2, l2_leaf_reg=15, iterations=150 (과적합 완화 튜닝)

In [ ]:
feature_cols = [
    '입점전_교체횟수', '입점전_공실횟수', '입점전_생존분기수',
    '층구분', '지하철거리_m', '건물내위치수', '연면적', '도로유형',
    '상권단계', 'TRDAR_SE_1', '공실_상권대비', '교체_상권대비',
    '경쟁자_수_입점시점', '경쟁자_증가율',
    'brand'
]
cat_features = ['층구분', '도로유형', '상권단계', 'TRDAR_SE_1', 'brand']

MODEL_PARAMS = dict(iterations=150, depth=2, learning_rate=0.05,
                    l2_leaf_reg=15, verbose=0, random_seed=42)

print(f'피처 수: {len(feature_cols)}개')
print(f'brand 카테고리: {sorted(df["brand"].unique())}')

In [ ]:
models = {}
score_rows = []

for year in [3, 5, 8]:
    n_q = year * 4
    sub = df[df['입점분기'].apply(
        lambda x: x in quarter_to_idx and quarter_to_idx[x] + n_q < len(quarters))].copy()
    sub['y'] = ((sub['생존분기수'] > n_q) |
                ((sub['현재영업중'] == 1) & (sub['생존분기수'] >= n_q))).astype(int)

    X, y = sub[feature_cols].copy(), sub['y']
    skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs, accs, f1s, tr_accs, te_accs = [], [], [], [], []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = CatBoostClassifier(cat_features=cat_features, **MODEL_PARAMS)
        m.fit(X_tr, y_tr)
        aucs.append(roc_auc_score(y_val, m.predict_proba(X_val)[:,1]))
        accs.append(accuracy_score(y_val, m.predict(X_val)))
        f1s.append(f1_score(y_val, m.predict(X_val)))
        tr_accs.append(m.score(X_tr, y_tr))
        te_accs.append(m.score(X_val, y_val))

    m_full = CatBoostClassifier(cat_features=cat_features, **MODEL_PARAMS)
    m_full.fit(X, y)
    models[year] = m_full

    gap = np.mean(tr_accs) - np.mean(te_accs)
    score_rows.append({
        '모델': f'{year}년차', '샘플': len(sub), '생존율': f'{y.mean():.1%}',
        'AUC': round(np.mean(aucs),3), 'Accuracy': round(np.mean(accs),3),
        'F1': round(np.mean(f1s),3),
        'Train': round(np.mean(tr_accs),3), 'Test': round(np.mean(te_accs),3),
        '과적합': '⚠️' if gap > 0.05 else '✅'
    })

print(pd.DataFrame(score_rows).set_index('모델').to_string())

## E-2. 브랜드별 OOF 예측력 점검 (참고용)
표본 적은 브랜드는 AUC가 불안정할 수 있음 → MVP 신뢰도 배지에 활용

In [ ]:
year = 5  # 대표로 5년차 확인
n_q = year*4
sub = df[df['입점분기'].apply(lambda x: x in quarter_to_idx and quarter_to_idx[x]+n_q < len(quarters))].copy()
sub['y'] = ((sub['생존분기수']>n_q) | ((sub['현재영업중']==1)&(sub['생존분기수']>=n_q))).astype(int)
X, y = sub[feature_cols].copy(), sub['y']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = np.zeros(len(sub))
for tr_idx, val_idx in skf.split(X, y):
    m = CatBoostClassifier(cat_features=cat_features, **MODEL_PARAMS)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof_pred[val_idx] = m.predict_proba(X.iloc[val_idx])[:,1]
sub['oof_pred'] = oof_pred

print(f"{'brand':<10} {'n':>5} {'실제율':>7} {'예측평균':>8} {'AUC':>6}")
for brand, g in sub.groupby('brand'):
    if g['y'].nunique()==2 and len(g)>=10:
        auc = f"{roc_auc_score(g['y'], g['oof_pred']):.3f}"
    else:
        auc = 'N/A'
    print(f"{brand:<10} {len(g):>5} {g['y'].mean():>7.1%} {g['oof_pred'].mean():>8.1%} {auc:>6}")

## E-3. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, year in zip(axes, [3,5,8]):
    fi = pd.Series(models[year].get_feature_importance(), index=feature_cols)\
         .sort_values(ascending=True).tail(10)
    fi.plot(kind='barh', ax=ax, color='#3B82F6')
    ax.set_title(f'{year}년차')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## F. 현재 공실 추출 + 가상입점분기 계산 (보정 B)
현재 공실(202512 NaN)의 입점전 변수를 "마지막 영업 시점 기준"으로 계산
→ 학습데이터(입점 직전 기준)와 분포 정합성 확보

In [ ]:
empty_ids = pivot[pivot['df_202512'].isna()].index.tolist()
empty_df = pd.DataFrame({'위치ID': empty_ids})
empty_df = empty_df.merge(viz, on='위치ID', how='left')
empty_df = empty_df.merge(result[['위치ID','TRDAR_CD','TRDAR_CD_N','상권평균교체','상권평균공실',
                                    '공실_상권대비','교체_상권대비']], on='위치ID', how='left')
empty_df = empty_df.dropna(subset=['위도','경도','시군구명'])
print(f'공실: {len(empty_df)}개')

In [ ]:
def calc_vacancy_history(위치ID):
    """가상입점분기 = 마지막 영업 분기 + 1, 그 이전 이력으로 변수 계산"""
    if 위치ID not in pivot.index:
        return np.nan, np.nan, np.nan, np.nan
    vals = pivot.loc[위치ID].values
    occupied_idx = [i for i, v in enumerate(vals) if pd.notna(v)]

    if not occupied_idx:
        return 0, 0, 0, quarter_nums[0]

    last_occ = max(occupied_idx)
    가상_idx = last_occ + 1
    가상입점분기 = quarter_nums[가상_idx] if 가상_idx < len(quarter_nums) else quarter_nums[-1]

    pre_q = quarters[:가상_idx]
    if not pre_q:
        return 0, 0, 0, 가상입점분기

    pre_vals = pivot.loc[위치ID][pre_q].values
    공실 = int(pd.isna(pre_vals).sum())
    생존 = int(pd.notna(pre_vals).sum())
    교체 = 0
    prev = pre_vals[0]
    for v in pre_vals[1:]:
        if pd.isna(prev) != pd.isna(v): 교체 += 1
        elif not pd.isna(prev) and not pd.isna(v) and prev != v: 교체 += 1
        prev = v
    return 교체, 공실, 생존, 가상입점분기

print('가상입점분기 기준 이력 계산 중...')
hist = empty_df['위치ID'].apply(calc_vacancy_history)
empty_df['입점전_교체횟수']   = [x[0] for x in hist]
empty_df['입점전_공실횟수']   = [x[1] for x in hist]
empty_df['입점전_생존분기수'] = [x[2] for x in hist]
empty_df['가상입점분기']      = [x[3] for x in hist]

print('\n=== 학습데이터 vs 공실(B) 분포 비교 ===')
for col in ['입점전_교체횟수','입점전_공실횟수','입점전_생존분기수']:
    print(f'{col}: 학습 {df[col].mean():.2f} / 공실 {empty_df[col].mean():.2f}')

## F-2. 결측 처리 + 상권단계 + 경쟁밀도 (가상입점분기 기준)

In [ ]:
for col in ['층구분','도로유형','TRDAR_SE_1']:
    empty_df[col] = empty_df[col].fillna('정보없음')
for col in ['지하철거리_m','건물내위치수','연면적']:
    empty_df[col] = empty_df[col].fillna(empty_df[col].median())
for col in ['상권평균교체','상권평균공실','공실_상권대비','교체_상권대비']:
    empty_df[col] = empty_df[col].fillna(0)

empty_df['입점period'] = empty_df['가상입점분기'].apply(to_period)
empty_df['상권단계']   = empty_df.apply(
    lambda r: get_stage(r['TRDAR_CD'], r['입점period']), axis=1)

empty_df = add_competitor_density(empty_df, q_col='가상입점분기')

print(empty_df['상권단계'].value_counts())

## G. 위치 점수 산출 (브랜드 무관 기본 추천)
`brand='일반카페'`로 고정해서 예측 → **"이 자리 자체가 얼마나 좋은가"**의 기준점

단조감소 제약(보정 A) 적용: `5년 ≤ 3년`, `8년 ≤ 5년`

In [ ]:
def predict_with_brand(target_df, brand_value):
    e = target_df.copy()
    e['brand'] = brand_value
    for year in [3, 5, 8]:
        e[f'생존확률_{year}년'] = models[year].predict_proba(e[feature_cols])[:,1]
    e['생존확률_5년'] = np.minimum(e['생존확률_5년'], e['생존확률_3년'])
    e['생존확률_8년'] = np.minimum(e['생존확률_8년'], e['생존확률_5년'])
    e['종합점수'] = (e['생존확률_3년']*0.3 + e['생존확률_5년']*0.4 + e['생존확률_8년']*0.3)
    return e

empty_general = predict_with_brand(empty_df, '일반카페')

non_mono = ((empty_general['생존확률_5년'] > empty_general['생존확률_3년']) |
             (empty_general['생존확률_8년'] > empty_general['생존확률_5년'])).mean()
print(f'비단조 비율: {non_mono:.1%}')
print(empty_general[['생존확률_3년','생존확률_5년','생존확률_8년','종합점수']].describe().round(3))

## H. 위치 점수 등급 + 저장

In [ ]:
def grade(s):
    if s >= 0.75: return 'A'
    if s >= 0.60: return 'B'
    if s >= 0.45: return 'C'
    return 'D'

empty_general['등급'] = empty_general['종합점수'].apply(grade)
print(empty_general['등급'].value_counts())

empty_general.to_csv('empty_scored_general.csv', index=False, encoding='utf-8-sig')
print('empty_scored_general.csv 저장 완료')

## I. 브랜드별 상위 100개 산출 (MVP용)
13개 프랜차이즈 브랜드 각각 `brand=해당브랜드`로 재예측 → 종합점수 상위 100개씩 추출
(모델 재학습 없이 predict만 다시 — 빠름)

In [ ]:
FRANCHISE_BRANDS = [
    '투썸플레이스','이디야','빽다방','공차','할리스','파스쿠찌','카페베네',
    '엔제리너스','탐앤탐스','폴바셋','컴포즈','메가커피','요거프레소'
]

all_top100 = []
for brand in FRANCHISE_BRANDS:
    e = predict_with_brand(empty_df, brand)
    top100 = e.sort_values('종합점수', ascending=False).head(100).copy()
    top100['brand'] = brand
    all_top100.append(top100)
    print(f'{brand}: 최고점수 {top100["종합점수"].max():.3f}, 100위 {top100["종합점수"].min():.3f}')

brand_top100 = pd.concat(all_top100, ignore_index=True)
print(f'\n전체: {len(brand_top100)}행')

## J. MVP용 데이터 저장 (csv + json)

In [ ]:
cols = ['위치ID','brand','도로명주소','시군구명','행정동명','층구분',
        '지하철거리_m','연면적','TRDAR_SE_1','TRDAR_CD_N','상권단계',
        '입점전_교체횟수','입점전_공실횟수','입점전_생존분기수',
        '생존확률_3년','생존확률_5년','생존확률_8년','종합점수','위도','경도']
out = brand_top100[cols].copy()

for c in ['생존확률_3년','생존확률_5년','생존확률_8년','종합점수']:
    out[c] = (out[c]*100).round(1)
out['지하철거리_m'] = out['지하철거리_m'].round(0).astype(int)
out['연면적']      = out['연면적'].round(0).astype(int)

out.to_csv('brand_top100.csv', index=False, encoding='utf-8-sig')
with open('brand_top100.json','w',encoding='utf-8') as f:
    json.dump(out.to_dict(orient='records'), f, ensure_ascii=False)

print(f'brand_top100.csv / json 저장 완료: {len(out)}행')